# Clan-Subnet Detector — GWMIN vs MORPH (method=auto)

Notebook de **calidad y latencia** del planificador de conjuntos independientes.

Objetivos:

1. Localizar / compilar `dump_morph` y fijar `MORPH_DUMP_BIN`.
2. Verificar correctitud (`is_independent`) para GWMIN y MORPH.
3. Comparar tamaño del conjunto independiente y tiempo de pared.
4. Ejecutar el pipeline de auditoría completo con `method=auto` y `method=gwmin`.
5. Guardar CSV + resumen Markdown.

**Entorno:** ejecute desde la raíz del ZIP `zero-trust-graph-engine/` (o ajuste `ZTGE_ROOT`).


In [ ]:
# CONFIG
import os, sys, time, json, platform, shutil, subprocess, tempfile
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

# Resolve ZTGE root (notebook may live next to the package or inside it)
CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/home/workdir/artifacts/zero-trust-graph-engine'),
    Path('/kaggle/working/zero-trust-graph-engine'),
    Path('/content/zero-trust-graph-engine'),
]
ZTGE_ROOT = None
for c in CANDIDATES:
    if (c / 'vendor' / 'clan_src' / 'clan_subnet_detector').is_dir():
        ZTGE_ROOT = c
        break
assert ZTGE_ROOT is not None, 'No se encontró zero-trust-graph-engine (vendor/clan_src)'

CLAN_SRC = ZTGE_ROOT / 'vendor' / 'clan_src'
sys.path.insert(0, str(CLAN_SRC))

OUT = Path.cwd() / 'clan_morph_vs_gwmin_results'
OUT.mkdir(parents=True, exist_ok=True)

MODE = 'full'  # 'quick' | 'full'
SEED = 42
if MODE == 'quick':
    N_LIST = [200, 500, 1000, 2000]
    REPS = 3
    BUDGET_MS = 50
else:
    N_LIST = [200, 500, 1000, 2000, 5000]
    REPS = 5
    BUDGET_MS = 100

print('ZTGE_ROOT =', ZTGE_ROOT)
print('Python', sys.version.split()[0], '|', platform.platform(), '| CPUs:', os.cpu_count())
print('MODE', MODE, '| N_LIST', N_LIST, '| BUDGET_MS', BUDGET_MS)


In [ ]:
# Locate or build dump_morph and set MORPH_DUMP_BIN
def ensure_dump_morph() -> Path:
    env = os.environ.get('MORPH_DUMP_BIN')
    if env and Path(env).is_file() and os.access(env, os.X_OK):
        return Path(env).resolve()
    candidates = [
        ZTGE_ROOT / 'bin' / 'dump_morph',
        ZTGE_ROOT / 'vendor' / 'morph_src' / 'dump_morph',
        Path('/tmp/dump_morph'),
        Path.cwd() / 'dump_morph',
    ]
    for p in candidates:
        if p.is_file():
            # Copy to /tmp to avoid noexec mounts (e.g. some cloud notebooks)
            dest = Path('/tmp/dump_morph')
            shutil.copy2(p, dest)
            dest.chmod(0o755)
            if os.access(dest, os.X_OK):
                return dest
    # Build from sources
    msrc = ZTGE_ROOT / 'vendor' / 'morph_src'
    src = msrc / 'dump_morph.cpp'
    solver = msrc / 'morph_qoblib_solver.cpp'
    assert src.is_file() and solver.is_file(), 'Faltan fuentes dump_morph.cpp / morph_qoblib_solver.cpp'
    dest = Path('/tmp/dump_morph')
    cmd = ['g++', '-O3', '-std=c++17', '-DNDEBUG', '-o', str(dest), str(src)]
    # compile from morph_src so #include works
    r = subprocess.run(cmd, cwd=str(msrc), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('Compilación de dump_morph falló:\n' + r.stderr)
    dest.chmod(0o755)
    return dest

DUMP = ensure_dump_morph()
os.environ['MORPH_DUMP_BIN'] = str(DUMP)
print('MORPH_DUMP_BIN =', DUMP)

from clan_subnet_detector.detection.morph_adapter import morph_status, morph_independent
from clan_subnet_detector.detection.independent_set import (
    plan_independent, gwmin_independent, is_independent,
)
from clan_subnet_detector.graph.graph import Graph
from clan_subnet_detector.audit.report import run_audit

st = morph_status()
print(json.dumps(st, indent=2))
assert st['available'], 'MORPH no disponible tras ensure_dump_morph'


In [ ]:
# Graph generators (ER / BA)
import random

def erdos_renyi(n: int, avg_deg: float, seed: int) -> Graph:
    rng = random.Random(seed)
    p = avg_deg / max(n - 1, 1)
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < p:
                edges.append((i, j))
    return Graph.from_edges(n, edges, directed=False)

def barabasi_albert(n: int, m: int, seed: int) -> Graph:
    rng = random.Random(seed)
    if n <= m:
        edges = [(i, j) for i in range(n) for j in range(i + 1, n)]
        return Graph.from_edges(n, edges, directed=False)
    edges = [(i, j) for i in range(m + 1) for j in range(i + 1, m + 1)]
    deg = [m] * (m + 1) + [0] * (n - m - 1)
    for new in range(m + 1, n):
        targets = set()
        total = sum(deg[:new])
        while len(targets) < m and total > 0:
            r = rng.random() * total
            acc = 0.0
            for v in range(new):
                acc += deg[v]
                if acc >= r:
                    targets.add(v)
                    break
        for t in targets:
            edges.append((new, t))
            deg[new] += 1
            deg[t] += 1
    return Graph.from_edges(n, edges, directed=False)

print('generators OK')


In [ ]:
# Benchmark: GWMIN vs MORPH (explicit) vs auto
AVG_DEG = 6
rows = []

def median(xs):
    s = sorted(xs)
    return s[len(s) // 2]

for family, builder in [
    ('ER', lambda n, seed: erdos_renyi(n, AVG_DEG, seed)),
    ('BA', lambda n, seed: barabasi_albert(n, max(2, AVG_DEG // 2), seed)),
]:
    for n in N_LIST:
        times = defaultdict(list)
        sizes = defaultdict(list)
        ok_flags = defaultdict(list)
        traces_auto = []
        m_edges = 0
        for r in range(REPS):
            seed = SEED + r * 1009 + n
            g = builder(n, seed)
            m_edges = g.edge_count()

            # GWMIN
            t0 = time.perf_counter()
            v_g = gwmin_independent(g)
            times['gwmin'].append((time.perf_counter() - t0) * 1000)
            sizes['gwmin'].append(len(v_g))
            ok_flags['gwmin'].append(is_independent(g, v_g))

            # MORPH explicit
            t0 = time.perf_counter()
            v_m = morph_independent(g, budget_ms=BUDGET_MS, seed=seed)
            times['morph'].append((time.perf_counter() - t0) * 1000)
            sizes['morph'].append(len(v_m))
            ok_flags['morph'].append(is_independent(g, v_m))

            # auto (should resolve to morph)
            tr = []
            t0 = time.perf_counter()
            v_a = plan_independent(g, method='auto', budget_ms=BUDGET_MS, seed=seed, trace_out=tr)
            times['auto'].append((time.perf_counter() - t0) * 1000)
            sizes['auto'].append(len(v_a))
            ok_flags['auto'].append(is_independent(g, v_a))
            traces_auto.append(tr[0] if tr else {})

        row = {
            'family': family,
            'n': n,
            'm': m_edges,
            'budget_ms': BUDGET_MS,
            'reps': REPS,
            'gwmin_ms_med': median(times['gwmin']),
            'morph_ms_med': median(times['morph']),
            'auto_ms_med': median(times['auto']),
            'gwmin_size_med': median(sizes['gwmin']),
            'morph_size_med': median(sizes['morph']),
            'auto_size_med': median(sizes['auto']),
            'gwmin_all_ok': all(ok_flags['gwmin']),
            'morph_all_ok': all(ok_flags['morph']),
            'auto_all_ok': all(ok_flags['auto']),
            'auto_resolved_morph': all(t.get('resolved') == 'morph' for t in traces_auto),
            'size_gain_morph_vs_gwmin_pct': 100 * (median(sizes['morph']) / max(median(sizes['gwmin']), 1) - 1),
        }
        rows.append(row)
        print(
            f"[{family}] n={n:<5d} m≈{m_edges:<6d}  "
            f"gwmin={row['gwmin_ms_med']:7.1f}ms/{row['gwmin_size_med']}  "
            f"morph={row['morph_ms_med']:7.1f}ms/{row['morph_size_med']}  "
            f"gain={row['size_gain_morph_vs_gwmin_pct']:+.1f}%  "
            f"auto→morph={row['auto_resolved_morph']} ok={row['morph_all_ok']}"
        )

perf = pd.DataFrame(rows)
perf.to_csv(OUT / 'gwmin_vs_morph_scaling.csv', index=False)
display(perf.round(2))


In [ ]:
# Full audit pipeline: method=gwmin vs method=auto on a fixed BA graph
g = barabasi_albert(2000, 3, seed=SEED)
print(f'Audit graph: n={g.n} m={g.edge_count()}')

t0 = time.perf_counter()
rep_g = run_audit(g, method='gwmin', budget_ms=BUDGET_MS, seed=SEED, independent_k=None)
t_g = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
rep_a = run_audit(g, method='auto', budget_ms=BUDGET_MS, seed=SEED, independent_k=None)
t_a = (time.perf_counter() - t0) * 1000

def ind_finding(rep):
    for f in rep.findings:
        if f.kind == 'independent_set':
            return f
    return None

fg, fa = ind_finding(rep_g), ind_finding(rep_a)
summary = {
    'gwmin_audit_ms': t_g,
    'auto_audit_ms': t_a,
    'gwmin_ind_size': fg.size if fg else None,
    'auto_ind_size': fa.size if fa else None,
    'gwmin_verified': fg.verified if fg else None,
    'auto_verified': fa.verified if fa else None,
    'auto_extra': fa.extra if fa else None,
    'gwmin_checks': rep_g.checks,
    'auto_checks': rep_a.checks,
}
print(json.dumps(summary, indent=2, default=str))
(OUT / 'audit_comparison.json').write_text(json.dumps(summary, indent=2, default=str))


In [ ]:
# Plots + final report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, family in zip(axes, ['ER', 'BA']):
    sub = perf[perf.family == family]
    if sub.empty:
        continue
    ax.plot(sub.n, sub.gwmin_ms_med, 'o-', label='GWMIN')
    ax.plot(sub.n, sub.morph_ms_med, 's-', label=f'MORPH @{BUDGET_MS}ms')
    ax.plot(sub.n, sub.auto_ms_med, 'd--', label='auto')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('n')
    ax.set_ylabel('mediana (ms)')
    ax.set_title(f'{family} — latencia planificador')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'latency_gwmin_vs_morph.png', dpi=150)
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, family in zip(axes2, ['ER', 'BA']):
    sub = perf[perf.family == family]
    if sub.empty:
        continue
    ax.plot(sub.n, sub.gwmin_size_med, 'o-', label='GWMIN size')
    ax.plot(sub.n, sub.morph_size_med, 's-', label='MORPH size')
    ax.set_xscale('log')
    ax.set_xlabel('n')
    ax.set_ylabel('tamaño mediana IS')
    ax.set_title(f'{family} — calidad (tamaño del IS)')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8)
fig2.tight_layout()
fig2.savefig(OUT / 'quality_gwmin_vs_morph.png', dpi=150)
plt.show()

checks = {
    'morph_available': bool(st.get('available')),
    'all_gwmin_verified': bool(perf.gwmin_all_ok.all()),
    'all_morph_verified': bool(perf.morph_all_ok.all()),
    'all_auto_verified': bool(perf.auto_all_ok.all()),
    'auto_always_resolved_morph': bool(perf.auto_resolved_morph.all()),
}
report = {
    'mode': MODE,
    'budget_ms': BUDGET_MS,
    'morph_binary': str(DUMP),
    'checks': checks,
    'mean_size_gain_pct': float(perf.size_gain_morph_vs_gwmin_pct.mean()),
    'table': perf.to_dict(orient='records'),
}
(OUT / 'comparison_report.json').write_text(json.dumps(report, indent=2, default=str))
md_lines = [
    '# GWMIN vs MORPH (method=auto) — Performance Report',
    '',
    f'- Mode: `{MODE}`',
    f'- Budget: {BUDGET_MS} ms',
    f'- Binary: `{DUMP}`',
    '',
    '## Automated checks',
]
for k, v in checks.items():
    md_lines.append(f'- `{k}`: **{"PASS" if v else "FAIL"}**')
md_lines += [
    '',
    f'- Mean size gain MORPH vs GWMIN: **{report["mean_size_gain_pct"]:.2f}%**',
    '',
    '## Scaling table',
    perf.round(2).to_markdown(index=False),
    '',
]
(OUT / 'COMPARISON_REPORT.md').write_text('\n'.join(md_lines))
print('Checks:', checks)
print('Mean size gain MORPH vs GWMIN: {:.2f}%'.format(report['mean_size_gain_pct']))
print('Artefacts in', OUT)
print(*[p.name for p in sorted(OUT.iterdir())], sep='\n- ')
